# NB04 — Are the coordinates actually usable? Coordinate frames and the BGC queries

**Goal.** Move from *has coordinates* to *can be joined*. A table can carry contig, start, end
and strand and still be useless for collocation if nothing else is expressed in the same frame.
This notebook tests frame compatibility and then demonstrates the two queries the user named:

- **(a) genes near a BGC region** — a genomic interval join
- **(b) multi-domain search** — including the stricter *domain order* variant

**Discriminating query (run first).** The identifier-overlap check between
`kescience.mgnify.gene_bgc.contig_id` and `kescience.mgnify.gene.contig_id`. If that
intersection is empty, the coordinates cannot support collocation no matter how complete they
look, and that is the most consequential negative result the project could produce. Per
`RESEARCH_PLAN.md` it runs before anything confirmatory.

In [1]:
import sys, os, json, time, re, subprocess
_T0 = time.time()
sys.path.insert(0, os.path.abspath("../src"))
import pandas as pd
from berdl_notebook_utils.setup_spark_session import get_spark_session
from census import CENSUS
from semcat import RX_SPAN, SEMCAT

spark = get_spark_session(app_name="acatlas_nb04")
pd.set_option("display.width", 240); pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60); pd.set_option("display.max_rows", 120)
os.makedirs("../figures", exist_ok=True)
cen = pd.DataFrame(CENSUS)

## 1. Discriminating query — do BGC regions and genes share a contig namespace?

`gene_bgc` is partitioned by `biome_id`, so the cheap and honest test is to pick one biome,
pull its distinct contig ids from both tables, and intersect. Restricting to a shared biome
avoids a false negative that would arise purely from comparing disjoint slices.

In [2]:
biomes = spark.sql('''
SELECT biome_id, COUNT(*) AS n FROM kescience.mgnify.gene_bgc
GROUP BY biome_id ORDER BY n DESC LIMIT 5''').toPandas()
print("biomes with the most BGC rows:")
print(biomes.to_string(index=False))
BIOME = biomes.biome_id.iloc[0]
print(f"\nusing biome_id = {BIOME}")

biomes with the most BGC rows:
biome_id     n
    soil 35354

using biome_id = soil


In [3]:
ov = spark.sql(f'''
WITH b AS (SELECT DISTINCT contig_id FROM kescience.mgnify.gene_bgc WHERE biome_id = '{BIOME}'),
     g AS (SELECT DISTINCT contig_id FROM kescience.mgnify.gene     WHERE biome_id = '{BIOME}')
SELECT (SELECT COUNT(*) FROM b) AS bgc_contigs,
       (SELECT COUNT(*) FROM g) AS gene_contigs,
       (SELECT COUNT(*) FROM b JOIN g USING (contig_id)) AS shared_contigs
''').toPandas()
print(ov.to_string(index=False))
bc, gc, sc = int(ov.bgc_contigs[0]), int(ov.gene_contigs[0]), int(ov.shared_contigs[0])
print()
print(f"BGC contigs matched to a gene contig: {sc}/{bc} = {sc/bc:.1%}" if bc else "no BGC contigs")
print()
if sc == 0:
    print("RESULT: NO SHARED CONTIG NAMESPACE -- the coordinates cannot support collocation.")
else:
    print("RESULT: the namespaces DO overlap, so a genomic-interval join is possible with no")
    print("mapping table. This is what makes the MGnify family usable for 'genes near a BGC'.")

 bgc_contigs  gene_contigs  shared_contigs
       28494       3416781           28494

BGC contigs matched to a gene contig: 28494/28494 = 100.0%

RESULT: the namespaces DO overlap, so a genomic-interval join is possible with no
mapping table. This is what makes the MGnify family usable for 'genes near a BGC'.


In [4]:
# Sanity check that the ids are literally the same strings, not merely similar counts.
print("example shared contig ids:")
ex = spark.sql(f'''
WITH b AS (SELECT DISTINCT contig_id FROM kescience.mgnify.gene_bgc WHERE biome_id = '{BIOME}'),
     g AS (SELECT DISTINCT contig_id FROM kescience.mgnify.gene     WHERE biome_id = '{BIOME}')
SELECT contig_id FROM b JOIN g USING (contig_id) LIMIT 5''').toPandas()
print(ex.to_string(index=False))

example shared contig ids:


        contig_id
 MGYG000503678_84
MGYG000503906_310
 MGYG000504150_47
MGYG000504367_173
MGYG000504438_272


## 2. Coordinate-frame map

Group the coordinate-bearing census tables by the frame their numbers are measured along. The
groups are coarse labels taken from the census's `coord_frame` text: two tables in the same
group are not thereby in the same sequence namespace (every isolated genome's contigs share
the label "own genome/contig"). Section 5 tests namespaces by query.

In [5]:
co = cen[cen.coords.isin(["yes", "packed"])][
    ["fq", "region_type", "coord_system", "coord_frame", "seq_ref", "strand"]].copy()
co["frame_group"] = co.coord_frame.fillna("UNKNOWN").map(lambda s: (
    "MGnify contig"            if "MGnify contig" in s else
    "pangenome cluster rep"    if "cluster representative" in s else
    "UniProt protein"          if "UniProt" in s else
    "PDB / structure"          if "PDB" in s else
    "own genome/contig (isolated)" if ("contig" in s or "chromosome" in s or "scaffold" in s) else
    "HMM / reference protein"  if ("HMM" in s or "reference protein" in s or "two frames" in s) else
    "UNRESOLVED (no sequence column)" if "UNKNOWN" in s else "other"))
print(co.groupby("frame_group").size().sort_values(ascending=False).to_string())
print()
for grp, sub in co.groupby("frame_group"):
    print(f"=== {grp} ({len(sub)} tables) ===")
    print(sub[["fq", "region_type", "coord_system", "strand"]].to_string(index=False))
    print()

frame_group
own genome/contig (isolated)    9
MGnify contig                   5
HMM / reference protein         3
UniProt protein                 2
other                           2
pangenome cluster rep           2

=== HMM / reference protein (3 tables) ===
                                       fq       region_type coord_system strand
         nmdc.results.pfam_annotation_gff    protein domain     multiple    NaN
   nmdc.results.annotation_kegg_orthology alignment segment     multiple    NaN
nmdc.results.annotation_enzyme_commission alignment segment     multiple    NaN

=== MGnify contig (5 tables) ===
                            fq               region_type coord_system      strand
     kescience.mgnify.gene_bgc biosynthetic gene cluster   nucleotide         NaN
  kescience.mgnify.gene_crispr              CRISPR array   nucleotide orientation
 kescience.mgnify.gene_defense            defense system   nucleotide         NaN
kescience.mgnify.gene_mobilome    mobile genetic element  

## 3. Query (a) — genes near a BGC region

The user's example. Executed on the MGnify frame, which section 1 showed is the only place this
is a single clean join. "Near" is parameterised by a flank so the query answers *inside or
within N kb of* a BGC, which is what BGC-boundary work actually needs.

In [6]:
FLANK = 5000
genes_near = spark.sql(f'''
SELECT b.bgc_id, b.nearest_mibig_class, b.contig_id,
       b.start_pos AS bgc_start, b.end_pos AS bgc_end,
       g.gene_id, g.start AS gene_start, g.end AS gene_end, g.strand,
       CASE WHEN g.start >= b.start_pos AND g.end <= b.end_pos THEN 'inside'
            ELSE 'flank' END AS placement
FROM kescience.mgnify.gene_bgc b
JOIN kescience.mgnify.gene g
  ON g.biome_id = b.biome_id AND g.contig_id = b.contig_id
 AND g.end   >= b.start_pos - {FLANK}
 AND g.start <= b.end_pos   + {FLANK}
WHERE b.biome_id = '{BIOME}'
ORDER BY b.bgc_id, g.start
''')
gn = genes_near.limit(400).toPandas()
print(f"returned {len(gn)} gene-BGC pairs (first 400) for biome {BIOME}")
print(gn.head(25).to_string(index=False))

returned 400 gene-BGC pairs (first 400) for biome soil
                     bgc_id nearest_mibig_class         contig_id  bgc_start  bgc_end             gene_id  gene_start  gene_end strand placement
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       8480    21413 MGYG000490806_00814        2775      3518      -     flank
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       8480    21413 MGYG000490806_00815        3515      3655      -     flank
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       8480    21413 MGYG000490806_00816        3688      4458      -     flank
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       8480    21413 MGYG000490806_00817        4448      5809      -     flank
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       8480    21413 MGYG000490806_00818        6276      6704      +     flank
MGYG000490806_101_sanntis_1          Saccharide MGYG000490806_101       848

In [7]:
# Scale of the result, and genes-per-BGC -- the number that tells you the join is sane.
stats = spark.sql(f'''
WITH j AS (
  SELECT b.bgc_id, g.gene_id,
         CASE WHEN g.start >= b.start_pos AND g.end <= b.end_pos THEN 1 ELSE 0 END AS inside
  FROM kescience.mgnify.gene_bgc b
  JOIN kescience.mgnify.gene g
    ON g.biome_id = b.biome_id AND g.contig_id = b.contig_id
   AND g.end >= b.start_pos - {FLANK} AND g.start <= b.end_pos + {FLANK}
  WHERE b.biome_id = '{BIOME}')
SELECT COUNT(*) AS gene_bgc_pairs, COUNT(DISTINCT bgc_id) AS bgcs_with_genes,
       SUM(inside) AS genes_inside,
       ROUND(COUNT(*) / COUNT(DISTINCT bgc_id), 1) AS genes_per_bgc,
       ROUND(SUM(inside) / COUNT(DISTINCT bgc_id), 1) AS genes_inside_per_bgc
FROM j''').toPandas()
print(stats.to_string(index=False))
print()
n_bgc = spark.sql(f"SELECT COUNT(*) c FROM kescience.mgnify.gene_bgc WHERE biome_id='{BIOME}'"
                  ).collect()[0][0]
print(f"BGCs in this biome: {n_bgc}; with >=1 gene found: {int(stats.bgcs_with_genes[0])} "
      f"({int(stats.bgcs_with_genes[0])/n_bgc:.0%})")
print()
print("A plausible genes-inside-per-BGC figure is the real check here: BGCs are typically a")
print("handful to a few dozen genes, so a number in that range means the interval join is")
print("operating in a genuinely shared coordinate frame, not just matching strings by luck.")

 gene_bgc_pairs  bgcs_with_genes  genes_inside  genes_per_bgc  genes_inside_per_bgc
         754313            35354        489978           21.3                  13.9

BGCs in this biome: 35354; with >=1 gene found: 35354 (100%)

A plausible genes-inside-per-BGC figure is the real check here: BGCs are typically a
handful to a few dozen genes, so a number in that range means the interval join is
operating in a genuinely shared coordinate frame, not just matching strings by luck.


In [8]:
gn.to_csv("../data/example_genes_near_bgc.csv", index=False)
print("wrote ../data/example_genes_near_bgc.csv")

# Add function to those genes -- the full use case is "which FUNCTIONS sit near a BGC".
func_near = spark.sql(f'''
SELECT b.nearest_mibig_class, e.cog_category, COUNT(*) AS n
FROM kescience.mgnify.gene_bgc b
JOIN kescience.mgnify.gene g
  ON g.biome_id = b.biome_id AND g.contig_id = b.contig_id
 AND g.end >= b.start_pos - {FLANK} AND g.start <= b.end_pos + {FLANK}
JOIN kescience.mgnify.gene_eggnog e
  ON e.gene_id = g.gene_id AND e.biome_id = g.biome_id
WHERE b.biome_id = '{BIOME}' AND e.cog_category IS NOT NULL AND e.cog_category <> '-'
GROUP BY b.nearest_mibig_class, e.cog_category
ORDER BY n DESC LIMIT 20''').toPandas()
print()
print("three-table join (BGC region -> gene coordinates -> eggNOG function):")
print(func_near.to_string(index=False))

wrote ../data/example_genes_near_bgc.csv



three-table join (BGC region -> gene coordinates -> eggNOG function):
nearest_mibig_class cog_category     n
         Saccharide            M 48873
         Saccharide            S 25588
         Polyketide            S 23844
              Other            S 18565
            Terpene            S 13905
              Other            M 13565
                NRP            S 13456
         Polyketide            I 12701
               RiPP            S 11803
         Polyketide           IQ  9702
         Polyketide            C  9359
                NRP            Q  9106
         Polyketide            Q  8571
            Terpene            I  8481
         Saccharide            E  8481
         Polyketide            K  7773
         Polyketide            E  7047
         Polyketide            M  7039
              Other            E  6984
         Saccharide            G  6521


## 4. Query (b) — multi-domain search

Two strengths of question, and BERDL supports them unequally:

1. **Co-occurrence** — "proteins carrying domains X *and* Y". Answerable from any
   coordinate-free table that lists domains per protein.
2. **Order and spacing** — "X then Y, N residues apart". Needs per-domain amino-acid
   coordinates, which only the span-bearing domain tables have.

Demonstrated on `kbase.ke_pangenome.interproscan_domains`, since it carries both Pfam
accessions and amino-acid spans on the same key.

In [9]:
# Co-occurrence: ABC transporter ATP-binding (PF00005) with a permease domain (PF00528).
PF_A, PF_B = "PF00005", "PF00528"
cooc = spark.sql(f'''
WITH d AS (
  SELECT gene_cluster_id, signature_acc, start, stop
  FROM kbase.ke_pangenome.interproscan_domains
  WHERE analysis = 'Pfam' AND signature_acc IN ('{PF_A}', '{PF_B}'))
SELECT COUNT(DISTINCT gene_cluster_id) AS proteins_with_both
FROM (SELECT gene_cluster_id FROM d GROUP BY gene_cluster_id
      HAVING COUNT(DISTINCT signature_acc) = 2)''').toPandas()
print(f"proteins carrying BOTH {PF_A} and {PF_B}: {int(cooc.proteins_with_both[0]):,}")

proteins carrying BOTH PF00005 and PF00528: 6,530


In [10]:
# Order + spacing: the query only coordinates make possible.
ordered = spark.sql(f'''
WITH a AS (SELECT gene_cluster_id, start AS a_start, stop AS a_stop, seq_len
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_A}'),
     b AS (SELECT gene_cluster_id, start AS b_start, stop AS b_stop
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_B}')
SELECT a.gene_cluster_id, a.seq_len, a.a_start, a.a_stop, b.b_start, b.b_stop,
       b.b_start - a.a_stop AS gap_aa,
       CASE WHEN a.a_stop < b.b_start THEN '{PF_A}->{PF_B}' ELSE '{PF_B}->{PF_A}' END AS domain_order
FROM a JOIN b USING (gene_cluster_id)
''')
# Aggregate over ALL pairs in Spark. A first version did ORDER BY gap_aa then LIMIT 300 and
# reported the median of that -- which is the extreme tail, not the distribution. Percentiles
# are computed over the full join; the CSV sample is taken WITHOUT an ordering.
agg = spark.sql(f'''
WITH a AS (SELECT gene_cluster_id, start AS a_start, stop AS a_stop
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_A}'),
     b AS (SELECT gene_cluster_id, start AS b_start, stop AS b_stop
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_B}'),
     j AS (SELECT a.gene_cluster_id, b.b_start - a.a_stop AS gap_aa,
                  CASE WHEN a.a_stop < b.b_start THEN 'A_then_B' ELSE 'B_then_A' END AS ord
           FROM a JOIN b USING (gene_cluster_id))
SELECT COUNT(*) AS pairs,
       COUNT(DISTINCT gene_cluster_id) AS proteins,
       SUM(CASE WHEN ord = 'A_then_B' THEN 1 ELSE 0 END) AS a_then_b,
       SUM(CASE WHEN ord = 'B_then_A' THEN 1 ELSE 0 END) AS b_then_a,
       percentile_approx(gap_aa, 0.25) AS p25,
       percentile_approx(gap_aa, 0.50) AS median_gap,
       percentile_approx(gap_aa, 0.75) AS p75
FROM j''').toPandas()
print(f"domain pairs over the FULL join ({PF_A} with {PF_B}):")
print(agg.to_string(index=False))
print()
ab, ba = int(agg.a_then_b[0]), int(agg.b_then_a[0])
print(f"{PF_A} before {PF_B}: {ab:,} ({ab/(ab+ba):.0%})")
print(f"{PF_B} before {PF_A}: {ba:,} ({ba/(ab+ba):.0%})")
print()
print("Reading this biologically: PF00528 is the ABC permease domain and PF00005 the")
print("ATP-binding domain. In fused transporters the permease is typically N-terminal and the")
print("ATPase C-terminal, so a majority of B-before-A is the expected architecture rather than")
print("an artifact -- which is itself a check that the amino-acid coordinates are meaningful.")

od = ordered.limit(300).toPandas()   # unordered sample for inspection
print()
print(f"inspection sample (unordered, {len(od)} rows):")
print(od.head(15).to_string(index=False))
od.to_csv("../data/example_multidomain_hits.csv", index=False)
print("wrote ../data/example_multidomain_hits.csv")

# PROTEIN-LEVEL order, deduplicated. A protein carrying repeat copies of either domain
# contributes several pairs to the figures above, so the pair-level percentage is
# pseudo-replicated -- the same class of error the schema_sig dedup in NB03 exists to avoid.
# One protein, one vote: a cluster counts as A_then_B only if the majority of its pairs are.
ordr = spark.sql(f'''
WITH a AS (SELECT gene_cluster_id, stop AS a_stop
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_A}'),
     b AS (SELECT gene_cluster_id, start AS b_start
           FROM kbase.ke_pangenome.interproscan_domains
           WHERE analysis = 'Pfam' AND signature_acc = '{PF_B}'),
     j AS (SELECT a.gene_cluster_id,
                  CASE WHEN a.a_stop < b.b_start THEN 1 ELSE 0 END AS is_ab
           FROM a JOIN b USING (gene_cluster_id)),
     per_protein AS (SELECT gene_cluster_id, AVG(is_ab) AS frac_ab, COUNT(*) AS n_pairs
                     FROM j GROUP BY gene_cluster_id)
SELECT COUNT(*) AS proteins,
       SUM(CASE WHEN frac_ab > 0.5 THEN 1 ELSE 0 END) AS protein_a_then_b,
       SUM(CASE WHEN frac_ab < 0.5 THEN 1 ELSE 0 END) AS protein_b_then_a,
       SUM(CASE WHEN frac_ab = 0.5 THEN 1 ELSE 0 END) AS protein_tied,
       SUM(CASE WHEN n_pairs > 1 THEN 1 ELSE 0 END) AS proteins_with_repeats
FROM per_protein''').toPandas()
print()
print("PROTEIN-LEVEL (one cluster, one vote):")
print(ordr.to_string(index=False))
pab, pba = int(ordr.protein_a_then_b[0]), int(ordr.protein_b_then_a[0])
n_prot = int(ordr.proteins[0])
print()
print(f"  {PF_B} before {PF_A}: {pba:,}/{n_prot:,} proteins = {pba/n_prot:.0%}")
print(f"  proteins contributing >1 pair (repeat domains): "
      f"{int(ordr.proteins_with_repeats[0]):,} ({int(ordr.proteins_with_repeats[0])/n_prot:.0%})")
print()
print("The pair-level and protein-level percentages are reported separately; the protein-level")
print("figure is the one to quote, for the same reason NB03 quotes per-schema_sig rates.")

domain pairs over the FULL join (PF00005 with PF00528):
 pairs  proteins  a_then_b  b_then_a  p25  median_gap  p75
  6963      6530       385      6578 -412        -392 -371

PF00005 before PF00528: 385 (6%)
PF00528 before PF00005: 6,578 (94%)

Reading this biologically: PF00528 is the ABC permease domain and PF00005 the
ATP-binding domain. In fused transporters the permease is typically N-terminal and the
ATPase C-terminal, so a majority of B-before-A is the expected architecture rather than
an artifact -- which is itself a check that the amino-acid coordinates are meaningful.



inspection sample (unordered, 300 rows):
      gene_cluster_id  seq_len  a_start  a_stop  b_start  b_stop  gap_aa     domain_order
    BPUT01000046.1_34      627      303     427       66     243    -361 PF00528->PF00005
 CAADSV010000012.1_11      671      371     520      108     286    -412 PF00528->PF00005
  CAAGKD010000130.1_3      462      220     358        2     165    -356 PF00528->PF00005
 CAAGPK010000036.1_13      396      371     395      108     286    -287 PF00528->PF00005
 CADAWB010000030.1_27      917      583     731       48     228    -683 PF00528->PF00005
  CAILJS010000207.1_1      569      299     436       26     207    -410 PF00528->PF00005
 CAJFKI010000052.1_11      540      309     465       87     224    -378 PF00528->PF00005
 CAJMMF010000017.1_28      458      297     413       81     250    -332 PF00528->PF00005
 CAJTGN010000019.1_21      503      307     445       70     244    -375 PF00528->PF00005
CAKAIQ010000002.1_474      470        5     141      288  


PROTEIN-LEVEL (one cluster, one vote):
 proteins  protein_a_then_b  protein_b_then_a  protein_tied  proteins_with_repeats
     6530               350              6167            13                    395

  PF00528 before PF00005: 6,167/6,530 proteins = 94%
  proteins contributing >1 pair (repeat domains): 395 (6%)

The pair-level and protein-level percentages are reported separately; the protein-level
figure is the one to quote, for the same reason NB03 quotes per-schema_sig rates.


In [11]:
# Why this query is impossible on the coordinate-free tables, stated concretely.
pf_str = spark.sql('''
SELECT query_name, PFAMs FROM kbase.ke_pangenome.eggnog_mapper_annotations
WHERE PFAMs LIKE '%ABC_tran%' AND PFAMs LIKE '%,%' LIMIT 5''').toPandas()
print("the same information in eggnog_mapper_annotations.PFAMs:")
print(pf_str.to_string(index=False))
print()
print("Domain NAMES in a comma-joined string: co-occurrence is answerable with LIKE, order and")
print("spacing are not recoverable at all. That is the practical difference between the two")
print("kinds of multi-domain search, and it is exactly what the coordinate columns buy.")

the same information in eggnog_mapper_annotations.PFAMs:
          query_name                               PFAMs
 JAIBHU010000197.1_6                      ABC_tran,Wzt_C
  NZ_LT629791.1_5220                    ABC_tran,DUF4162
CAAFCT010000001.1_17                    ABC_tran,DUF4162
 CAKQCP010000045.1_5 ABC_membrane,ABC_tran,Peptidase_C39
JAEXUX010000016.1_46               ABC_tran,ABC_tran_Xtn

Domain NAMES in a comma-joined string: co-occurrence is answerable with LIKE, order and
spacing are not recoverable at all. That is the practical difference between the two
kinds of multi-domain search, and it is exactly what the coordinate columns buy.


## 5. Joinability, checked by query

**Population.** Every census table that is *coordinate-bearing*, meaning it has a positional
span: start+end, start+length, or a packed location (NB02 states the rule). That is 23
tables. `refdata.microagrobiome.bgc_catalog`, a BGC table with no sequence coordinates, is
listed for reference and is outside the 23.

**Rule.** A table is *joinable to genes by position* when an interval predicate in SQL places
its rows against gene rows on the same nucleotide sequence identifier, and a query in this
section shows it doing so. Three routes count:

- **direct**: a region table and a gene table in the same database share the sequence
  identifier, and an interval join (±5 kb) returns genes.
- **self**: the table's rows are themselves genes (census `region_type` starts with "gene"),
  so the table is its own gene partner. A self-join on a hash sample of whole contigs must
  return neighbouring genes within ±5 kb. A span derived from the table's own columns
  (start+length, or a parsed JSON location) still counts, and the note says so.
- **bridged**: the table gets its sequence identifier from one bridge table that maps each
  row to exactly one sequence. The 1:1 property is checked on a hash bucket of the key,
  and the self-join is run through the bridge.

Two verdicts are not joins:

- **protein-space only**: the census records amino-acid or HMM-model coordinates
  (`coord_system` amino_acid or multiple). These positions are not on a nucleotide
  sequence, so no interval predicate can place them against genes. This verdict applies the
  rule to NB02's coordinate-system evidence. No join query is run, and the table marks it
  as judgement.
- **no gene table**: a nucleotide region table whose database holds no other table with a
  span. Checked by listing the database's tables and columns.

Two tables that are joinable through a third table's genes (e.g. two MGnify region tables
via `gene`) are not counted twice. Each table gets one verdict.

In [12]:
# 5a. direct: MGnify region tables against kescience.mgnify.gene.
# Region tables are small (<200k rows), so these run over every row, not a sample.
checks = {}

MGNIFY_REGIONS = [
    ("kescience.mgnify.gene_bgc",      "bgc_id",            "start_pos",    "end_pos"),
    ("kescience.mgnify.gene_crispr",   "crispr_id",         "crispr_start", "crispr_end"),
    ("kescience.mgnify.gene_defense",  "defense_system_id", "system_start", "system_end"),
    ("kescience.mgnify.gene_mobilome", "element_id",        "start_pos",    "end_pos"),
]
mg_rows = []
for t, rid, s, e in MGNIFY_REGIONS:
    r = spark.sql(f'''
    WITH reg  AS (SELECT DISTINCT biome_id, contig_id, {rid} AS rid, {s} AS s, {e} AS e FROM {t}),
         rc   AS (SELECT DISTINCT biome_id, contig_id FROM reg),
         gc   AS (SELECT DISTINCT biome_id, contig_id FROM kescience.mgnify.gene),
         near AS (SELECT reg.rid, COUNT(DISTINCT g.gene_id) AS n
                  FROM reg LEFT JOIN kescience.mgnify.gene g
                    ON g.biome_id = reg.biome_id AND g.contig_id = reg.contig_id
                   AND g.`end` >= reg.s - {FLANK} AND g.start <= reg.e + {FLANK}
                  GROUP BY reg.rid)
    SELECT (SELECT COUNT(*) FROM rc)                                      AS region_contigs,
           (SELECT COUNT(*) FROM rc JOIN gc USING (biome_id, contig_id))  AS shared_contigs,
           (SELECT COUNT(*) FROM near)                                    AS regions,
           (SELECT COUNT(*) FROM near WHERE n > 0)                        AS regions_with_gene,
           (SELECT percentile_approx(n, 0.5) FROM near)                   AS median_genes,
           (SELECT COUNT(*) FROM reg WHERE s < 0 OR e < 0)                AS negative_coord_rows
    ''').toPandas().iloc[0]
    mg_rows.append(dict(table=t, partner="kescience.mgnify.gene", **{k: int(v) for k, v in r.items()}))
mg = pd.DataFrame(mg_rows)
print(f"join key (biome_id, contig_id); window: region interval +/- {FLANK:,} bp")
print(mg.to_string(index=False))
for r in mg.itertuples():
    checks[r.table] = dict(
        route="direct", partner=r.partner,
        passed=bool(r.shared_contigs > 0 and r.regions_with_gene > 0),
        evidence=(f"{r.shared_contigs:,}/{r.region_contigs:,} region contigs found in gene; "
                  f"{r.regions_with_gene:,}/{r.regions:,} regions have a gene within +/-5 kb "
                  f"(median {r.median_genes})"))

join key (biome_id, contig_id); window: region interval +/- 5,000 bp
                         table               partner  region_contigs  shared_contigs  regions  regions_with_gene  median_genes  negative_coord_rows
     kescience.mgnify.gene_bgc kescience.mgnify.gene           28494           28494    35354              35354            18                    0
  kescience.mgnify.gene_crispr kescience.mgnify.gene           34623           34392    42437              42185             8                    0
 kescience.mgnify.gene_defense kescience.mgnify.gene           10864           10864    13739              13734            12                    0
kescience.mgnify.gene_mobilome kescience.mgnify.gene           49160           49160   171373             171373            11                    2


In [13]:
# 5b. self: gene tables placed against their own genes. Each check takes a hash sample of
# whole sequences (pmod(xxhash64(seq), M) = 0), so every gene on a sampled sequence is
# present, and counts the other genes within +/-5 kb of each gene. Sequence ids are
# qualified by genome/organism where the table's contig ids are only unique within one.
content_tbls = sorted(r[1] for r in spark.sql("SHOW TABLES IN kbase.content").collect()
                      if r[1].endswith("_features"))
# kbase.content location = [[contig, start, strand, length]]; on '-' features `start` is the
# rightmost base (KBase convention). Only the first location segment is used.
content_src = " UNION ALL ".join(f'''
  SELECT concat('{t}', '|', loc[0][0]) AS seq, id AS gid,
         CASE WHEN loc[0][2] = '-' THEN CAST(loc[0][1] AS BIGINT) - CAST(loc[0][3] AS BIGINT) + 1
              ELSE CAST(loc[0][1] AS BIGINT) END AS s,
         CASE WHEN loc[0][2] = '-' THEN CAST(loc[0][1] AS BIGINT)
              ELSE CAST(loc[0][1] AS BIGINT) + CAST(loc[0][3] AS BIGINT) - 1 END AS e
  FROM (SELECT id, from_json(location, 'array<array<string>>') AS loc FROM kbase.content.{t})''' for t in content_tbls)

SELF = [
    ("kescience.mgnify.gene", "self", 65536, "",
     "SELECT concat(biome_id,'|',contig_id) AS seq, gene_id AS gid, start AS s, `end` AS e FROM kescience.mgnify.gene"),
    ("refdata.fungal_db.pombase_gene_coords", "self", 1, "",
     "SELECT chromosome AS seq, gene_id AS gid, start AS s, `end` AS e FROM refdata.fungal_db.pombase_gene_coords"),
    ("kescience.fitnessbrowser.gene", "self", 64, "begin/end are strings, cast to BIGINT",
     "SELECT concat(orgId,'|',scaffoldId) AS seq, locusId AS gid, CAST(begin AS BIGINT) AS s, CAST(`end` AS BIGINT) AS e FROM kescience.fitnessbrowser.gene"),
    ("globalusers.carbon_source_phenotypes.gene_calls_table", "self", 1024, "start/end are strings, cast to BIGINT",
     "SELECT concat(genomeid,'|',contigid) AS seq, geneid AS gid, CAST(start AS BIGINT) AS s, CAST(`end` AS BIGINT) AS e FROM globalusers.carbon_source_phenotypes.gene_calls_table"),
    ("refdata.jgi_mycocosm.features", "self", 4096, "no end column: end derived as start + length_bp - 1",
     "SELECT concat(genome_id,'|',contig_id) AS seq, feature_id AS gid, start AS s, start + length_bp - 1 AS e FROM refdata.jgi_mycocosm.features"),
    ("refdata.jgi_virus.metavr_protein", "self", 65536, "",
     "SELECT scaffold_oid AS seq, gene_oid AS gid, start AS s, `end` AS e FROM refdata.jgi_virus.metavr_protein"),
    ("refdata.bvbrc.bvbrc_feature_fact", "self", 1048576, "start_pos/end_pos are strings, cast to BIGINT",
     "SELECT concat(genome_id,'|',contig) AS seq, feature_id AS gid, CAST(start_pos AS BIGINT) AS s, CAST(end_pos AS BIGINT) AS e FROM refdata.bvbrc.bvbrc_feature_fact"),
    ("kbase.content.c_<genome>_features", "self", 1, f"span parsed from the JSON `location` column; {len(content_tbls)} genome tables",
     content_src),
    ("kbase.genomes.feature", "bridged", 1048576, "contig_id from kbase.genomes.contig_x_feature",
     "SELECT x.contig_id AS seq, f.feature_id AS gid, f.start AS s, f.`end` AS e FROM kbase.genomes.contig_x_feature x JOIN kbase.genomes.feature f USING (feature_id)"),
]

def self_join(src, M):
    return spark.sql(f'''
    WITH g AS (SELECT * FROM ({src}) WHERE seq IS NOT NULL AND s IS NOT NULL AND e IS NOT NULL
                                       AND pmod(abs(xxhash64(seq)), {M}) = 0),
         n AS (SELECT a.seq, a.gid, a.s, a.e, COUNT(b.gid) AS nb
               FROM g a LEFT JOIN g b
                 ON b.seq = a.seq AND b.gid <> a.gid
                AND b.e >= a.s - {FLANK} AND b.s <= a.e + {FLANK}
               GROUP BY a.seq, a.gid, a.s, a.e)
    SELECT COUNT(DISTINCT seq) AS seqs_sampled, COUNT(*) AS genes_sampled,
           SUM(CASE WHEN nb > 0 THEN 1 ELSE 0 END) AS genes_with_neighbour,
           percentile_approx(nb, 0.5) AS median_neighbours,
           SUM(CASE WHEN e < s THEN 1 ELSE 0 END) AS end_before_start
    FROM n''').toPandas().iloc[0]

self_rows = []
for t, route, M, note, src in SELF:
    t1 = time.time()
    r = self_join(src, M)
    self_rows.append(dict(table=t, route=route, sample=f"1/{M}", seconds=round(time.time() - t1),
                          **{k: int(v) for k, v in r.items()}))
    print(f"  {t}: {time.time() - t1:.0f}s")
sj = pd.DataFrame(self_rows)
print()
print(sj.to_string(index=False))
for (t, route, M, note, src), r in zip(SELF, sj.itertuples()):
    checks[t] = dict(
        route=route, partner="itself" if route == "self" else "itself, via kbase.genomes.contig_x_feature",
        passed=bool(r.genes_sampled > 0 and r.genes_with_neighbour > 0),
        evidence=(f"sample 1/{M} of sequences: {r.seqs_sampled:,} sequences, {r.genes_sampled:,} genes, "
                  f"{r.genes_with_neighbour:,} with a neighbour within +/-5 kb (median {r.median_neighbours})"
                  + (f"; {note}" if note else "")))

  kescience.mgnify.gene: 10s


  refdata.fungal_db.pombase_gene_coords: 2s


  kescience.fitnessbrowser.gene: 2s


  globalusers.carbon_source_phenotypes.gene_calls_table: 2s


  refdata.jgi_mycocosm.features: 2s


  refdata.jgi_virus.metavr_protein: 53s


  refdata.bvbrc.bvbrc_feature_fact: 302s


  kbase.content.c_<genome>_features: 3s


  kbase.genomes.feature: 258s

                                                table   route    sample  seconds  seqs_sampled  genes_sampled  genes_with_neighbour  median_neighbours  end_before_start
                                kescience.mgnify.gene    self   1/65536       10           139           1230                  1217                  7                 0
                refdata.fungal_db.pombase_gene_coords    self       1/1        2             6          12685                 12684                 12                 0
                        kescience.fitnessbrowser.gene    self      1/64        2             4          11660                 11660                 10                 0
globalusers.carbon_source_phenotypes.gene_calls_table    self    1/1024        2            98           9156                  9126                 10                 0
                        refdata.jgi_mycocosm.features    self    1/4096        2           212           7383               

In [14]:
# 5c. pombase: the gene table against its sibling CDS and exon tables (same chromosome
# names; each CDS/exon should fall inside the gene with the same gene_id).
for sib in ["cds", "exon"]:
    r = spark.sql(f'''
    WITH g AS (SELECT DISTINCT chromosome FROM refdata.fungal_db.pombase_gene_coords),
         c AS (SELECT DISTINCT chromosome FROM refdata.fungal_db.pombase_{sib}_coords),
         inside AS (SELECT c.gene_id, c.start, c.`end`, MAX(g.gene_id IS NOT NULL) AS hit
                    FROM refdata.fungal_db.pombase_{sib}_coords c
                    LEFT JOIN refdata.fungal_db.pombase_gene_coords g
                      ON g.chromosome = c.chromosome AND g.gene_id = c.gene_id
                     AND c.start >= g.start AND c.`end` <= g.`end`
                    GROUP BY c.gene_id, c.start, c.`end`)
    SELECT (SELECT COUNT(*) FROM c) AS sib_chromosomes,
           (SELECT COUNT(*) FROM c JOIN g USING (chromosome)) AS shared_chromosomes,
           (SELECT COUNT(*) FROM inside) AS sib_intervals,
           (SELECT COUNT(*) FROM inside WHERE hit) AS inside_own_gene''').toPandas().iloc[0]
    print(f"pombase_{sib}_coords: {int(r.shared_chromosomes)}/{int(r.sib_chromosomes)} chromosomes shared; "
          f"{int(r.inside_own_gene):,}/{int(r.sib_intervals):,} {sib} intervals lie inside their own gene")
    checks["refdata.fungal_db.pombase_gene_coords"]["evidence"] += (
        f"; {int(r.inside_own_gene):,}/{int(r.sib_intervals):,} {sib} intervals inside their own gene")

pombase_cds_coords: 6/6 chromosomes shared; 5,134/5,134 cds intervals lie inside their own gene


pombase_exon_coords: 6/6 chromosomes shared; 18,025/18,026 exon intervals lie inside their own gene


In [15]:
# 5d. The kbase.genomes bridge, bounded. A full COUNT(DISTINCT feature_id) over ~1e9 rows is
# avoided. Instead every feature_id in hash bucket pmod(xxhash64(feature_id), 1024) = 0 is
# taken from both tables. Because the bucket is a function of feature_id, all copies of an
# id fall in the same bucket, so "rows == distinct ids" inside the bucket is an exact
# uniqueness test for those ids, and the overlap is an exact containment test for them.
t1 = time.time()
BUCKET = "pmod(abs(xxhash64(feature_id)), 1024) = 0"
br = spark.sql(f'''
WITH x AS (SELECT feature_id, contig_id FROM kbase.genomes.contig_x_feature WHERE {BUCKET}),
     f AS (SELECT feature_id FROM kbase.genomes.feature WHERE {BUCKET})
SELECT (SELECT COUNT(*) FROM kbase.genomes.contig_x_feature)        AS bridge_rows_total,
       (SELECT COUNT(*) FROM kbase.genomes.feature)                 AS feature_rows_total,
       (SELECT COUNT(*) FROM kbase.genomes.functional_annotation_rast) AS rast_rows_total,
       (SELECT COUNT(*) FROM x)                                      AS bucket_bridge_rows,
       (SELECT COUNT(DISTINCT feature_id) FROM x)                    AS bucket_bridge_ids,
       (SELECT COUNT(*) FROM f)                                      AS bucket_feature_rows,
       (SELECT COUNT(DISTINCT feature_id) FROM f)                    AS bucket_feature_ids,
       (SELECT COUNT(*) FROM (SELECT DISTINCT feature_id FROM x) a
                        JOIN (SELECT DISTINCT feature_id FROM f) b USING (feature_id)) AS bucket_shared_ids
''').toPandas().iloc[0]
br = {k: int(v) for k, v in br.items()}
for k, v in br.items(): print(f"  {k:22s} {v:>15,}")
bridge_1to1 = (br["bucket_bridge_rows"] == br["bucket_bridge_ids"] == br["bucket_feature_ids"]
               == br["bucket_feature_rows"] == br["bucket_shared_ids"] > 0)
print()
print(f"method: hash bucket 1/1024 of feature_id ({br['bucket_bridge_ids']:,} ids); {time.time()-t1:.0f}s")
print(f"each bucket feature_id appears once in contig_x_feature and once in feature, "
      f"and the two id sets are identical: {bridge_1to1}")
c = checks["kbase.genomes.feature"]
c["passed"] = c["passed"] and bridge_1to1
c["evidence"] = (f"bridge 1:1 on a 1/1024 feature_id bucket ({br['bucket_bridge_ids']:,} ids unique in both "
                 f"tables, sets identical: {bridge_1to1}); " + c["evidence"])

  bridge_rows_total        1,011,650,903
  feature_rows_total       1,011,650,903
  rast_rows_total            253,173,194
  bucket_bridge_rows             987,681
  bucket_bridge_ids              987,681
  bucket_feature_rows            987,681
  bucket_feature_ids             987,681
  bucket_shared_ids              987,681

method: hash bucket 1/1024 of feature_id (987,681 ids); 449s
each bucket feature_id appears once in contig_x_feature and once in feature, and the two id sets are identical: True


In [16]:
# 5e. refdata.smag.bgc: is there any other table in refdata.smag with a span to place genes on?
smag_rows = []
for tbl in sorted(r[1] for r in spark.sql("SHOW TABLES IN refdata.smag").collect()):
    cols = [r["col_name"] for r in spark.sql(f"DESCRIBE refdata.smag.{tbl}").collect()
            if r["col_name"] and not r["col_name"].startswith("#")]
    span_cols = [c for c in cols if re.search(RX_SPAN, c.lower())]
    smag_rows.append(dict(table=tbl, n_columns=len(cols), span_columns=", ".join(span_cols)))
smag = pd.DataFrame(smag_rows)
print(smag.to_string(index=False))
others_with_span = smag[(smag.table != "bgc") & (smag.span_columns != "")]
smag_no_gene_table = len(others_with_span) == 0
print()
print(f"tables in refdata.smag other than bgc with a span column: {len(others_with_span)}")
checks["refdata.smag.bgc"] = dict(
    route="no gene table", partner="(none in refdata.smag)", passed=smag_no_gene_table,
    evidence=f"{len(smag)} tables in refdata.smag; none besides bgc has a span column")

                 table  n_columns span_columns
                   bgc         10   start, end
          cas_proteins         16             
        cpr_dpann_mags         52             
  mag_quality_taxonomy         53             
        mag_sample_map         34             
            pangenomes          5             
    sample_environment         43             
       sample_metadata         14             
sra_recovered_metadata         18             
            virus_host         12             

tables in refdata.smag other than bgc with a span column: 0


In [17]:
# 5f. Independent evidence from SemCat. Single-column verified joins live in `edges`
# (verified = true); `verified_joins` holds composite-key joins only. List the verified edges
# on the sequence/feature keys used above. Also run the SemCat CLI's join-keys for the two
# pairs the CLI was asked about (the CLI reads its own local catalog, whose version it prints).
ev = spark.sql(f'''
SELECT src, dst, evidence_source, join_tier, containment_src_in_dst
FROM {SEMCAT}.edges
WHERE verified AND (
      src RLIKE '^kescience[.]mgnify[.]gene_(bgc|crispr|defense|mobilome)[.]contig_id$'
   OR src RLIKE '^refdata[.]fungal_db[.]pombase_.*[.](gene_id|chromosome)$'
   OR src RLIKE '^kbase[.]genomes[.](feature|contig_x_feature)[.]feature_id$'
   OR dst RLIKE '^kbase[.]genomes[.](feature|contig_x_feature)[.]feature_id$')
ORDER BY src''').toPandas()
print(f"verified SemCat edges on these keys: {len(ev)}")
print(ev.to_string(index=False))
print()
for a, b in [("kescience.mgnify.gene_bgc", "kescience.mgnify.gene"),
             ("kbase.genomes.feature", "kbase.genomes.contig_x_feature")]:
    try:
        out = subprocess.run(["semcat", "join-keys", a, b], capture_output=True, text=True, timeout=600)
        txt = "\n".join(l for l in (out.stdout + out.stderr).splitlines()
                        if "HF_TOKEN" not in l and "Fetching" not in l)
    except Exception as e:
        txt = f"semcat CLI not run: {e}"
    print(f"$ semcat join-keys {a} {b}")
    print(txt.strip())
    print()
import shutil
try:
    v = subprocess.run(["semcat", "--version"], capture_output=True, text=True, timeout=120)
    print(f"semcat executable: {shutil.which('semcat')}")
    print(f"semcat --version: {(v.stdout + v.stderr).strip()}")
except Exception as e:
    print(f"semcat CLI not available: {e}")

verified SemCat edges on these keys: 6
                                          src                                           dst evidence_source join_tier  containment_src_in_dst
          kescience.mgnify.gene_bgc.contig_id               kescience.mgnify.gene.contig_id   structural_fk  verified                  1.0000
       kescience.mgnify.gene_crispr.contig_id               kescience.mgnify.gene.contig_id   structural_fk  verified                  0.9933
      kescience.mgnify.gene_defense.contig_id               kescience.mgnify.gene.contig_id   structural_fk  verified                  1.0000
     kescience.mgnify.gene_mobilome.contig_id               kescience.mgnify.gene.contig_id   structural_fk  verified                  1.0000
 refdata.fungal_db.pombase_cds_coords.gene_id refdata.fungal_db.pombase_gene_coords.gene_id   structural_fk  verified                  1.0000
refdata.fungal_db.pombase_exon_coords.gene_id refdata.fungal_db.pombase_gene_coords.gene_id   structural_fk  

$ semcat join-keys kescience.mgnify.gene_bgc kescience.mgnify.gene
candidate join keys: kescience.mgnify.gene_bgc  ⋈  kescience.mgnify.gene   (3 column pair(s))
  contig_id  =  contig_id   [VERIFIED, fwd 1.00 · rev 0.00 · one-way, verified]
  contig_id  =  contig_id   [value-join·UNVERIFIED, fwd 1.00 · rev 0.01 · one-way, meaningful]
  genome_id  =  genome_id   [value-join·UNVERIFIED, fwd 0.99 · rev 0.09 · one-way, meaningful]
[semcat] STALE/SHADOW INSTALL: 3 distinct semcat versions on the path (orphaned editable pointer or stacked install). Fix (keep it EDITABLE — a wheel would freeze a stale catalog): bash /home/mamillerpa/semcat/scripts/reinstall.sh



$ semcat join-keys kbase.genomes.feature kbase.genomes.contig_x_feature
candidate join keys: kbase.genomes.feature  ⋈  kbase.genomes.contig_x_feature   (1 column pair(s))
  feature_id  =  feature_id   [value-join·UNVERIFIED, fwd 1.00 · rev 1.00 · clean, meaningful]
[semcat] STALE/SHADOW INSTALL: 3 distinct semcat versions on the path (orphaned editable pointer or stacked install). Fix (keep it EDITABLE — a wheel would freeze a stale catalog): bash /home/mamillerpa/semcat/scripts/reinstall.sh

semcat executable: /opt/conda/bin/semcat
semcat --version: [semcat] STALE/SHADOW INSTALL: 3 distinct semcat versions on the path (orphaned editable pointer or stacked install). Fix (keep it EDITABLE — a wheel would freeze a stale catalog): bash /home/mamillerpa/semcat/scripts/reinstall.sh
usage: semcat [-h] [--remote]
              {build,publish,update-repo,refresh,skills,info,where,capabilities,help,doctor,resolve,resolve-tables,explain,feasible,ground-set,quantity,features,tables,databases,join

In [18]:
# 5g. The verdict table, derived from the checks above plus the census's coordinate system.
# `evidence` says whether each verdict rests on a query in this notebook or on the rule.
ver = []
for r in co.itertuples():
    role = "gene" if str(r.region_type).startswith("gene") else "region"
    if r.coord_system in ("amino_acid", "multiple"):
        v, joinable, basis, note = ("protein-space only", False, "judgement: rule applied to NB02 coordinate system",
                                    f"{r.coord_system} coordinates on {r.coord_frame}")
        partner = "(none: not a nucleotide frame)"
    elif r.fq in checks:
        c = checks[r.fq]
        v = c["route"] if c["passed"] else f"{c['route']} (check FAILED)"
        joinable = c["passed"] and c["route"] in ("direct", "self", "bridged")
        basis, note, partner = "query (this notebook, section 5)", c["evidence"], c["partner"]
    else:
        v, joinable, basis, note, partner = "NOT CHECKED", False, "none", "", ""
    ver.append(dict(region_table=r.fq, role=role, coord_system=r.coord_system, frame=r.coord_frame,
                    gene_partner=partner, collocation=v, joinable=joinable, evidence=basis, note=note))
ver.append(dict(region_table="refdata.microagrobiome.bgc_catalog", role="region", coord_system="none",
                frame="none (latitude/longitude only)", gene_partner="(n/a)", collocation="no coordinates",
                joinable=False, evidence="census (NB02); outside the 23 coordinate-bearing tables",
                note="BGC table a researcher would reach for; listed for reference"))
jd = pd.DataFrame(ver)
assert not (jd.collocation == "NOT CHECKED").any(), jd[jd.collocation == "NOT CHECKED"]
assert not jd.collocation.str.contains("FAILED").any(), jd[jd.collocation.str.contains("FAILED")]
jd.to_csv("../data/bgc_joinability.csv", index=False)

pop = jd[jd.collocation != "no coordinates"]
print(f"coordinate-bearing tables assessed: {len(pop)} "
      f"({(pop.role == 'region').sum()} region tables, {(pop.role == 'gene').sum()} gene tables)")
print()
print(pd.crosstab(pop.collocation, pop.role, margins=True).to_string())
print()
print(f"joinable to genes by position: {int(pop.joinable.sum())} of {len(pop)}  "
      f"(region tables {int(pop[pop.role=='region'].joinable.sum())} of {(pop.role=='region').sum()}; "
      f"gene tables {int(pop[pop.role=='gene'].joinable.sum())} of {(pop.role=='gene').sum()})")
print(f"verdicts resting on a query: {(pop.evidence.str.startswith('query')).sum()}; "
      f"on judgement (rule + NB02 coordinate system): {(pop.evidence.str.startswith('judgement')).sum()}")
print()
print(jd[["region_table", "role", "collocation", "joinable", "evidence"]].to_string(index=False))
print()
print("wrote ../data/bgc_joinability.csv")

coordinate-bearing tables assessed: 23 (14 region tables, 9 gene tables)

role                gene  region  All
collocation                          
bridged                1       0    1
direct                 0       4    4
no gene table          0       1    1
protein-space only     0       9    9
self                   8       0    8
All                    9      14   23

joinable to genes by position: 13 of 23  (region tables 4 of 14; gene tables 9 of 9)
verdicts resting on a query: 14; on judgement (rule + NB02 coordinate system): 9

                                         region_table   role        collocation  joinable                                                evidence
              kbase.ke_pangenome.interproscan_domains region protein-space only     False       judgement: rule applied to NB02 coordinate system
                kbase.ke_pangenome.bakta_pfam_domains region protein-space only     False       judgement: rule applied to NB02 coordinate system
                

In [19]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

order = ["direct", "bridged", "self", "no gene table", "protein-space only", "no coordinates"]
colour = {"direct": "#2e7d32", "bridged": "#43a047", "self": "#81c784",
          "no gene table": "#c62828", "protein-space only": "#1f5fa8", "no coordinates": "#6d4c41"}
missing = set(jd.collocation) - set(colour)
assert not missing, f"no colour for {missing}"
jd2 = jd.copy()
jd2["rank"] = jd2.collocation.map({k: i for i, k in enumerate(order)})
jd2 = jd2.sort_values(["rank", "role", "region_table"])

fig, ax = plt.subplots(figsize=(13, 9))
y = list(range(len(jd2)))
ax.barh(y, [1] * len(jd2), color=[colour[c] for c in jd2.collocation])
ax.set_yticks(y)
ax.set_yticklabels([f"{r.region_table}  ({r.role})" for r in jd2.itertuples()], fontsize=8)
ax.set_xticks([]); ax.set_xlim(0, 1)
for i, r in enumerate(jd2.itertuples()):
    tag = r.collocation + ("" if r.evidence.startswith("query") else
                           "  [rule, not a join query]" if r.evidence.startswith("judgement") else "  [census]")
    ax.text(0.015, i, tag, va="center", ha="left", fontsize=8, color="white", fontweight="bold")
_nb01 = json.load(open("../data/nb01_summary.json"))
n_pop = int((jd.collocation != "no coordinates").sum())
ax.set_title("Can this table be placed against genes by position?\n"
             f"{int(jd.joinable.sum())} of {n_pop} coordinate-bearing tables joinable "
             f"(green); lakehouse read {time.strftime('%Y-%m-%d')}", fontsize=12)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("../figures/coordinate_frame_map.png", dpi=150, bbox_inches="tight")
print("wrote ../figures/coordinate_frame_map.png")
plt.show()

wrote ../figures/coordinate_frame_map.png


In [20]:
pop = jd[jd.collocation != "no coordinates"]
summary = dict(
    biome_tested=str(BIOME),
    bgc_contigs=bc, gene_contigs=gc, shared_contigs=sc,
    shared_contig_frac=float(sc / bc) if bc else None,
    gene_bgc_pairs=int(stats.gene_bgc_pairs[0]),
    bgcs_with_genes=int(stats.bgcs_with_genes[0]),
    bgcs_in_biome=int(n_bgc),
    genes_per_bgc=float(stats.genes_per_bgc[0]),
    genes_inside_per_bgc=float(stats.genes_inside_per_bgc[0]),
    proteins_with_both_pfams=int(cooc.proteins_with_both[0]),
    multidomain_pairs_total=int(agg.pairs[0]),
    multidomain_a_then_b=int(agg.a_then_b[0]),
    multidomain_b_then_a=int(agg.b_then_a[0]),
    multidomain_proteins=int(ordr.proteins[0]),
    multidomain_protein_a_then_b=int(ordr.protein_a_then_b[0]),
    multidomain_protein_b_then_a=int(ordr.protein_b_then_a[0]),
    multidomain_proteins_with_repeats=int(ordr.proteins_with_repeats[0]),
    median_interdomain_gap_aa=float(agg.median_gap[0]),
    interdomain_gap_p25=float(agg.p25[0]), interdomain_gap_p75=float(agg.p75[0]),
    coordinate_bearing_assessed=int(len(pop)),
    region_tables=int((pop.role == "region").sum()),
    gene_tables=int((pop.role == "gene").sum()),
    joinable_total=int(pop.joinable.sum()),
    joinable_region_tables=int(pop[pop.role == "region"].joinable.sum()),
    joinable_gene_tables=int(pop[pop.role == "gene"].joinable.sum()),
    verdict_direct=int((pop.collocation == "direct").sum()),
    verdict_self=int((pop.collocation == "self").sum()),
    verdict_bridged=int((pop.collocation == "bridged").sum()),
    verdict_protein_space=int((pop.collocation == "protein-space only").sum()),
    verdict_no_gene_table=int((pop.collocation == "no gene table").sum()),
    verdicts_by_query=int(pop.evidence.str.startswith("query").sum()),
    verdicts_by_judgement=int(pop.evidence.str.startswith("judgement").sum()),
    bridge_rows_total=br["bridge_rows_total"], feature_rows_total=br["feature_rows_total"],
    rast_rows_total=br["rast_rows_total"],
    bridge_bucket_ids=br["bucket_bridge_ids"], bridge_1to1_in_bucket=bool(bridge_1to1),
    semcat_verified_edges_on_join_keys=int(len(ev)),
    notebook_seconds=round(time.time() - _T0),
)
bad = {k: type(v).__name__ for k, v in summary.items()
       if not isinstance(v, (int, float, str, bool, type(None)))}
if bad: print("NON-SERIALIZABLE:", bad)
json.dump(summary, open("../data/nb04_summary.json", "w"), indent=2, default=str)
print(json.dumps(summary, indent=2, default=str))
print(f"NB04 wall time: {summary['notebook_seconds']} s")

{
  "biome_tested": "soil",
  "bgc_contigs": 28494,
  "gene_contigs": 3416781,
  "shared_contigs": 28494,
  "shared_contig_frac": 1.0,
  "gene_bgc_pairs": 754313,
  "bgcs_with_genes": 35354,
  "bgcs_in_biome": 35354,
  "genes_per_bgc": 21.3,
  "genes_inside_per_bgc": 13.9,
  "proteins_with_both_pfams": 6530,
  "multidomain_pairs_total": 6963,
  "multidomain_a_then_b": 385,
  "multidomain_b_then_a": 6578,
  "multidomain_proteins": 6530,
  "multidomain_protein_a_then_b": 350,
  "multidomain_protein_b_then_a": 6167,
  "multidomain_proteins_with_repeats": 395,
  "median_interdomain_gap_aa": -392.0,
  "interdomain_gap_p25": -412.0,
  "interdomain_gap_p75": -371.0,
  "coordinate_bearing_assessed": 23,
  "region_tables": 14,
  "gene_tables": 9,
  "joinable_total": 13,
  "joinable_region_tables": 4,
  "joinable_gene_tables": 9,
  "verdict_direct": 4,
  "verdict_self": 8,
  "verdict_bridged": 1,
  "verdict_protein_space": 9,
  "verdict_no_gene_table": 1,
  "verdicts_by_query": 14,
  "verdicts_b